# Fase 3 — Fine-tuning QLoRA de `product-analyst-agent` y ablación

Entrena un adaptador LoRA sobre `Qwen/Qwen2.5-7B-Instruct` con el dataset sintético (`scripts/build_dataset.py`), evalúa base vs. base+LoRA en el conjunto de prueba (requerimientos nunca vistos) y publica el adaptador en Hugging Face Hub.

**Runtime → GPU (T4 alcanza con r=16 y secuencias de 3 072 tokens; L4/A100 acelera).** Sin API keys de proveedores; solo `HF_TOKEN` para publicar el adaptador.

In [ ]:
!git clone https://github.com/javierrosado/tesis.git /content/tesis
%cd /content/tesis
!pip -q install -r requirements.txt
!pip -q install torch transformers accelerate bitsandbytes peft trl datasets outlines

In [ ]:
import os, json, sys
sys.path.insert(0, '/content/tesis')
# os.environ['HF_TOKEN'] = '...'   # necesario solo para push_to_hub
HUB_REPO = 'javierrosado/product-analyst-lora'
BASE = 'Qwen/Qwen2.5-7B-Instruct'
SEED = 2026

## 1. Dataset (sintético, por reglas, partición por requerimiento)

In [ ]:
!python scripts/build_dataset.py --variants 12 --seed 2026
!cat data/datasets/product-analyst/README.md

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files={'train':'data/datasets/product-analyst/train.jsonl','val':'data/datasets/product-analyst/val.jsonl','test':'data/datasets/product-analyst/test.jsonl'})
ds = ds.map(lambda ex: {'messages': [{'role': m['role'], 'content': m['content']} for m in ex['messages']]})
print(ds)

## 2. Modelo base en int4 + LoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
tok = AutoTokenizer.from_pretrained(BASE); tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])

## 3. Entrenamiento (SFT sobre el turno del asistente)

In [ ]:
from trl import SFTTrainer, SFTConfig
cfg = SFTConfig(output_dir='outputs/pa-lora', num_train_epochs=2, per_device_train_batch_size=1, gradient_accumulation_steps=8,
                learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03, logging_steps=5, eval_strategy='epoch', save_strategy='epoch',
                max_length=3072, packing=False, bf16=False, fp16=True, seed=SEED, report_to='none', assistant_only_loss=True)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds['train'], eval_dataset=ds['val'], processing_class=tok, peft_config=lora)
trainer.train()
trainer.save_model('outputs/pa-lora/final')
print(trainer.state.log_history[-3:])

## 4. Ablación: base vs. base+LoRA sobre el test (requerimientos no vistos)

Misma decodificación (greedy, 1 800 tokens máx.) y mismas métricas F5 evaluadas en código (`core/eval_dataset.py`).

In [ ]:
from core.registry import Registry
from core.eval_dataset import evaluate_predictions
from peft import PeftModel
reg = Registry('/content/tesis'); agent = reg.agents['product-analyst-agent']
test = [json.loads(l) for l in open('data/datasets/product-analyst/test.jsonl')]

def generate(m, ex):
    prompt = tok.apply_chat_template(ex['messages'][:2], tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors='pt').to(m.device)
    with torch.no_grad():
        out = m.generate(**ids, max_new_tokens=1800, do_sample=False)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
pred_base = [generate(base, ex) for ex in test]
res_base = evaluate_predictions(agent, test, pred_base)
lora_model = PeftModel.from_pretrained(base, 'outputs/pa-lora/final')
pred_lora = [generate(lora_model, ex) for ex in test]
res_lora = evaluate_predictions(agent, test, pred_lora)
import pandas as pd
tabla = pd.DataFrame({'base (sin LoRA)': res_base['aggregate'], 'base + LoRA': res_lora['aggregate']})
print(tabla)
json.dump({'base': res_base, 'lora': res_lora}, open('outputs/ablacion_product_analyst.json','w'), ensure_ascii=False, indent=1)

## 5. Publicar el adaptador y activarlo en el pipeline

In [ ]:
# trainer.model.push_to_hub(HUB_REPO, private=True)
# tok.push_to_hub(HUB_REPO, private=True)
print('''Luego, en config/providers.yaml:
overrides:
  product-analyst-agent:
    provider: hf_local
    adapter_weights: %s
''' % HUB_REPO)

## 6. Rerun del vertical con el agente entrenado
Repite `notebooks/run_pilot.ipynb` con el override activo. La telemetría registrará `model_id = Qwen/...+javierrosado/product-analyst-lora` para ese agente y el base para los demás; la comparación de rondas de gobernanza y rechazos de contrato entre ambas corridas es la evidencia de 3.1.4.